In [11]:
import requests
import pandas as pd

# URL API SoilGrids
url = "https://rest.isric.org/soilgrids/v2.0/properties/query"

# Coordonnées des pays
countries = {
    "Benin": (9.3, 2.3),
    "Senegal": (14.5, -14.5),
    "Ghana": (7.9, -1.0),
    "Nigeria": (9.1, 8.7),
    "Mali": (17.0, -4.0),
    "Burkina Faso": (12.3, -1.5),
    "Cote dIvoire": (7.5, -5.5)
}

# Liste finale
all_data = []

# Boucle sur les pays
for country, (lat, lon) in countries.items():

    print(f"Collecte des données pour {country}...")

    # Dictionnaire pour stocker les données du pays
    row = {
        "country": country
    }

    # Liste des propriétés à récupérer
    properties = {
        "phh2o": "soil_ph",
        "soc": "organic_matter",
        "clay": "clay_pct"
    }

    # Boucle sur les variables
    for prop, column_name in properties.items():

        params = {
            "lon": lon,
            "lat": lat,
            "property": prop,
            "depth": "0-5cm",
            "value": "mean"
        }

        response = requests.get(url, params=params)

        data = response.json()

        try:
            value = data["properties"]["layers"][0]["depths"][0]["values"]["mean"]

            # Conversion spéciale pour le pH
            if prop == "phh2o":
                value = value / 10

            row[column_name] = value

        except Exception as e:
            print(f"Erreur pour {country} - {prop} :", e)
            row[column_name] = None

    # Ajout du pays à la liste finale
    all_data.append(row)

# Création DataFrame final
soil_df = pd.DataFrame(all_data)

# Affichage
print("\nDataset sol final :")
print(soil_df)

# Sauvegarde CSV
soil_df.to_csv("soil.csv", index=False)

print("\nsoil.csv créé avec succès.")

Collecte des données pour Benin...
Collecte des données pour Senegal...
Collecte des données pour Ghana...
Collecte des données pour Nigeria...
Collecte des données pour Mali...
Collecte des données pour Burkina Faso...
Collecte des données pour Cote dIvoire...

Dataset sol final :
        country  soil_ph  organic_matter  clay_pct
0         Benin      6.2             242       167
1       Senegal      6.6              77       195
2         Ghana      6.2             247       273
3       Nigeria      5.8             294       211
4          Mali      7.8              69       310
5  Burkina Faso      6.3              86       163
6  Cote dIvoire      6.3             304       215

soil.csv créé avec succès.
